# **Penting**
- Jangan mengubah atau menambahkan cell text yang sudah disediakan, Anda hanya perlu mengerjakan cell code yang sudah disediakan.
- Pastikan seluruh kriteria memiliki output yang sesuai, karena jika tidak ada output dianggap tidak selesai.
- Misal, Anda menggunakan df = df.dropna() silakan gunakan df.isnull().sum() sebagai tanda sudah berhasil. Silakan sesuaikan seluruh output dengan perintah yang sudah disediakan.
- Pastikan Anda melakukan Run All sebelum mengirimkan submission untuk memastikan seluruh cell berjalan dengan baik.
- Pastikan Anda menggunakan variabel df dari awal sampai akhir dan tidak diperbolehkan mengganti nama variabel tersebut.
- Hapus simbol pagar (#) pada kode yang bertipe komentar jika Anda menerapkan kriteria tambahan
- Biarkan simbol pagar (#) jika Anda tidak menerapkan kriteria tambahan
- Pastikan Anda mengerjakan sesuai section yang sudah diberikan tanpa mengubah judul atau header yang disediakan.

# **1. Import Library**
Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [20]:
# Semua import untuk klasifikasi
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

# **2. Memuat Dataset dari Hasil Clustering**
Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [21]:
# Gunakan dataset hasil clustering yang memiliki fitur Target
# Silakan gunakan dataset data_clustering jika tidak menerapkan Interpretasi Hasil Clustering [Advanced]
# Silakan gunakan dataset data_clustering_inverse jika menerapkan Interpretasi Hasil Clustering [Advanced]
# Lengkapi kode berikut
# ___ = pd_read_csv("___.csv")
df = pd.read_csv('data_clustering_inverse.csv') if os.path.exists('data_clustering_inverse.csv') else pd.read_csv('data_clustering.csv')
df.head(1)

,TransactionAmount,TransactionDate,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,TransactionAmount_bin,TransactionAmount_bin_label,AccountBalance_bin,AccountBalance_bin_label,Target
0,14.09,2023-04-11 16:29:14,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,2024-11-04 08:08:08,Sangat Rendah,1,Tinggi,3,2


In [22]:
# Tampilkan 5 baris pertama dengan function head.
df.head()

,TransactionAmount,TransactionDate,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,TransactionAmount_bin,TransactionAmount_bin_label,AccountBalance_bin,AccountBalance_bin_label,Target
0,14.09,2023-04-11 16:29:14,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,2024-11-04 08:08:08,Sangat Rendah,1,Tinggi,3,2
1,376.24,2023-06-27 16:44:19,Debit,Houston,ATM,68.0,Doctor,141.0,1.0,13758.91,2024-11-04 08:09:35,Tinggi,3,Sangat Tinggi,2,0
2,126.29,2023-07-10 18:16:08,Debit,Mesa,Online,19.0,Student,56.0,1.0,1122.35,2024-11-04 08:07:04,Rendah,0,Sangat Rendah,1,3
3,184.50,2023-05-05 16:32:11,Debit,Raleigh,Online,26.0,Student,25.0,1.0,8569.06,2024-11-04 08:09:06,Rendah,0,Sangat Tinggi,2,3
4,13.45,2023-10-16 17:51:24,Debit,Atlanta,Online,45.0,Student,198.0,1.0,7429.40,2024-11-04 08:06:39,Sangat Rendah,1,Tinggi,3,1


# **3. Data Splitting**
Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [23]:
# Pisahkan fitur dan target
X = df.drop(columns=['Target'], errors='ignore')
y = df['Target'].astype(int)

# Identifikasi kolom kategorikal dan numerik
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Handle kolom tanggal - ekstraksi fitur waktu
date_cols = [c for c in X.columns if 'Date' in c or 'date' in c]
for col in date_cols:
    if col in X.columns:
        try:
            dt = pd.to_datetime(X[col], errors='coerce')
            X[f'{col}_hour'] = dt.dt.hour.fillna(0).astype(int)
            X[f'{col}_dayofweek'] = dt.dt.dayofweek.fillna(0).astype(int)
            X[f'{col}_month'] = dt.dt.month.fillna(0).astype(int)
            X = X.drop(columns=[col])
            if col in categorical_cols:
                categorical_cols.remove(col)
        except Exception:
            pass

# Encode kolom kategorikal yang tersisa
label_encoders = {}
for col in categorical_cols:
    if col in X.columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Features: {X.columns.tolist()}")

Training set size: (2011, 19)
Test set size: (503, 19)
Features: ['TransactionAmount', 'TransactionType', 'Location', 'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration', 'LoginAttempts', 'AccountBalance', 'TransactionAmount_bin', 'TransactionAmount_bin_label', 'AccountBalance_bin', 'AccountBalance_bin_label', 'TransactionDate_hour', 'TransactionDate_dayofweek', 'TransactionDate_month', 'PreviousTransactionDate_hour', 'PreviousTransactionDate_dayofweek', 'PreviousTransactionDate_month']


# **4. Membangun Model Klasifikasi**
Setelah memilih algoritma klasifikasi yang sesuai, langkah selanjutnya adalah melatih model menggunakan data latih.

Berikut adalah rekomendasi tahapannya.
1. Menggunakan algoritma klasifikasi yaitu Decision Tree.
2. Latih model menggunakan data yang sudah dipisah.

In [25]:
# Buatlah model klasifikasi menggunakan Decision Tree
clf_dt = DecisionTreeClassifier(random_state=42)
clf_dt.fit(X_train, y_train)
y_pred_dt = clf_dt.predict(X_test)

print("Decision Tree Model trained successfully")
print(f"Training Accuracy: {clf_dt.score(X_train, y_train):.4f}")
print(f"Test Accuracy: {clf_dt.score(X_test, y_test):.4f}")

Decision Tree Model trained successfully
Training Accuracy: 1.0000
Test Accuracy: 1.0000


In [26]:
# Menyimpan Model
import joblib
joblib.dump(clf_dt, 'decision_tree_model.h5')

['decision_tree_model.h5']

# **5. Memenuhi Kriteria Skilled dan Advanced dalam Membangun Model Klasifikasi**



**Biarkan kosong jika tidak menerapkan kriteria skilled atau advanced**

In [27]:
# Melatih model menggunakan algoritma klasifikasi scikit-learn selain Decision Tree.
# Logistic Regression
clf_lr = LogisticRegression(max_iter=1000, n_jobs=None)
clf_lr.fit(X_train, y_train)
y_pred_lr = clf_lr.predict(X_test)

# Random Forest
clf_rf = RandomForestClassifier(n_estimators=200, random_state=42)
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)

{"LR_acc": accuracy_score(y_test, y_pred_lr), "RF_acc": accuracy_score(y_test, y_pred_rf)}

{'LR_acc': 0.852882703777336, 'RF_acc': 1.0}

In [28]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada seluruh algoritma yang sudah dibuat.
models = {
    'DecisionTree': (clf_dt, y_pred_dt),
    'LogisticRegression': (clf_lr, y_pred_lr),
    'RandomForest': (clf_rf, y_pred_rf)
}

report = {}
for name, (model, y_pred) in models.items():
    report[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    }
report

{'DecisionTree': {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
 'LogisticRegression': {'accuracy': 0.852882703777336,
  'precision': 0.8568705635223068,
  'recall': 0.852882703777336,
  'f1': 0.8541658591160113},
 'RandomForest': {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}}

In [29]:
# Menyimpan Model Selain Decision Tree
# Model ini bisa lebih dari satu
import joblib
joblib.dump(clf_lr, 'explore_LogisticRegression_classification.h5')
joblib.dump(clf_rf, 'explore_RandomForest_classification.h5')

['explore_RandomForest_classification.h5']

Hyperparameter Tuning Model

Pilih salah satu algoritma yang ingin Anda tuning

In [31]:
# Lakukan Hyperparameter Tuning dan Latih ulang.
# Lakukan dalam satu cell ini saja.
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid.fit(X_train, y_train)

,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [32]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.
y_pred_best = grid.best_estimator_.predict(X_test)
{
    'best_params': grid.best_params_,
    'accuracy': accuracy_score(y_test, y_pred_best),
    'precision': precision_score(y_test, y_pred_best, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_pred_best, average='weighted', zero_division=0),
    'f1': f1_score(y_test, y_pred_best, average='weighted', zero_division=0)
}

{'best_params': {'max_depth': None,
  'min_samples_leaf': 1,
  'min_samples_split': 2},
 'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0}

In [33]:
# Menyimpan Model hasil tuning
import joblib
joblib.dump(grid.best_estimator_, 'tuning_classification.h5')

['tuning_classification.h5']

End of Code